## Vision-Authority-Resources Triangle Workflow
In this workflow, three LLM roles sit at the corners of a triangle. Each one holds a different kind of power over a plan, and no corner can finish the plan alone:

```
                 VISION
          (what we want, and why)
               /        \
              /          \
             /   PLAN     \
            /              \
   RESOURCES ------------ AUTHORITY
 (what we actually have)   (who may say yes)
```

- **Vision** proposes the goal, the success criteria and the steps. It has no budget and cannot approve its own plan.
- **Resources** checks the plan against what actually exists (budget, people, time, tools). It reports the allocation, the gaps and the cost. It does not judge whether the goal is worth pursuing.
- **Authority** holds the decision rights. It checks the plan and the allocation against the mandate and policy, then returns `APPROVED`, `REVISE` or `REJECTED`. It does not write plans or allocate resources.

When Authority returns `REVISE`, the feedback from both Resources and Authority goes back to Vision, and the loop runs again.

### When to use this workflow
This workflow fits decisions where a good idea is not enough on its own: it also has to be affordable and allowed. Good signs of fit:

- The ambition, the constraints and the approval rules are separate concerns that a single prompt tends to blur together.
- You want a record of *why* a plan was approved, with the cost and the policy checks it passed.
- Constraints are explicit enough to write down (a budget, a headcount, a policy document).

Compared with the [Evaluator-Optimizer](evaluator_optimizer.ipynb) loop, this pattern splits the evaluator into two independent checks, feasibility (Resources) and legitimacy (Authority). The generator can't satisfy one by quietly trading away the other.

In [ ]:
from util import llm_call, extract_xml

MODEL = "claude-sonnet-5-5"

def vision(prompt: str, goal: str, feedback: str = "") -> str:
    """Propose (or revise) a plan for the goal."""
    context = f"\n<feedback>\n{feedback}\n</feedback>" if feedback else ""
    response = llm_call(f"{prompt}\n<goal>\n{goal}\n</goal>{context}", model=MODEL)
    plan = extract_xml(response, "plan")

    print("\n=== VISION ===")
    print(plan)
    return plan

def resources(prompt: str, plan: str, inventory: str) -> tuple[str, str]:
    """Check the plan against the available resources."""
    response = llm_call(
        f"{prompt}\n<inventory>\n{inventory}\n</inventory>\n<plan>\n{plan}\n</plan>",
        model=MODEL,
    )
    allocation = extract_xml(response, "allocation")
    gaps = extract_xml(response, "gaps")

    print("=== RESOURCES ===")
    print(f"Allocation:\n{allocation}\n")
    print(f"Gaps:\n{gaps}")
    return allocation, gaps

def authority(prompt: str, plan: str, allocation: str, gaps: str, mandate: str) -> tuple[str, str]:
    """Decide whether the plan, as resourced, may go ahead."""
    response = llm_call(
        f"{prompt}\n<mandate>\n{mandate}\n</mandate>\n<plan>\n{plan}\n</plan>"
        f"\n<allocation>\n{allocation}\n</allocation>\n<gaps>\n{gaps}\n</gaps>",
        model=MODEL,
    )
    decision = extract_xml(response, "decision").strip()
    rationale = extract_xml(response, "rationale")

    print("=== AUTHORITY ===")
    print(f"Decision: {decision}")
    print(f"Rationale:\n{rationale}\n")
    return decision, rationale

def triangle(goal: str, inventory: str, mandate: str, prompts: dict, max_rounds: int = 3) -> dict:
    """Run Vision -> Resources -> Authority until Authority approves, rejects, or rounds run out."""
    feedback = ""
    history = []

    for round_num in range(1, max_rounds + 1):
        print(f"\n########## ROUND {round_num} ##########")
        plan = vision(prompts["vision"], goal, feedback)
        allocation, gaps = resources(prompts["resources"], plan, inventory)
        decision, rationale = authority(prompts["authority"], plan, allocation, gaps, mandate)

        history.append({"round": round_num, "plan": plan, "allocation": allocation,
                        "gaps": gaps, "decision": decision, "rationale": rationale})

        if decision in ("APPROVED", "REJECTED"):
            break

        feedback = (f"Resources reported these gaps:\n{gaps}\n\n"
                    f"Authority asked for revision because:\n{rationale}")

    return {"final": history[-1], "history": history}

### Example Use Case: Launching an internal AI pilot
Vision wants to roll out an AI support assistant. Resources knows the real budget and staffing. Authority applies the company's approval policy, which Vision never sees directly. Vision only learns about the policy through Authority's feedback, just as it would in a real organisation.

In [ ]:
vision_prompt = """
You are VISION. You set direction: what should be achieved, why it matters,
and how success will be measured. You do not control budget and you cannot approve
your own plan. If there is feedback from Resources or Authority, revise the plan to
address it while keeping as much of the original intent as possible.

Output in this format:
<plan>
Goal: ...
Why it matters: ...
Success criteria: (measurable)
Steps: (numbered, each with the resources it needs: money, people, time, tools)
Changes from previous version: (if revising)
</plan>
"""

resources_prompt = """
You are RESOURCES. You know exactly what is available, listed in <inventory>.
Check the plan step by step against it. Do not judge whether the goal is worthwhile;
only judge whether it can be resourced. Be precise with numbers.

Output in this format:
<allocation>
For each step: what is allocated from the inventory, and the running total of budget,
people and time.
</allocation>
<gaps>
Each shortfall or over-commitment, with the size of the gap. Write "None" if fully resourced.
</gaps>
"""

authority_prompt = """
You are AUTHORITY. You hold the decision rights defined in <mandate>. You do not write
plans or allocate resources. Check the plan and its allocation against every rule in
the mandate, and treat unresolved resource gaps as blocking.

- APPROVED: every mandate rule is satisfied and there are no blocking gaps.
- REVISE: the plan could be made acceptable; say exactly what must change.
- REJECTED: the goal itself is outside the mandate.

Output in this format:
<decision>APPROVED, REVISE, or REJECTED</decision>
<rationale>
Each mandate rule and whether it passes, then the required changes if any.
</rationale>
"""

goal = """
Deploy an AI assistant that answers customer support tickets for our software product,
so customers get faster answers and the support team can focus on hard cases.
"""

inventory = """
- Budget: $40,000 for this quarter
- Engineers: 2 (each available 50% of their time)
- Support agents available for review and labelling: 3
- Timeline: 10 weeks remaining in the quarter
- Existing tools: ticketing system with an API, a 2-year archive of resolved tickets
"""

mandate = """
1. Spend above $50,000 needs board approval; this committee can approve up to $50,000.
2. No customer-facing AI output may go out without a human-review phase of at least 4 weeks first.
3. Customer data must not leave approved vendors; a data-handling review is required before launch.
4. Every project must name a single accountable owner.
"""

prompts = {"vision": vision_prompt, "resources": resources_prompt, "authority": authority_prompt}
result = triangle(goal, inventory, mandate, prompts)

### Reading the decision record
`result["history"]` holds every round: the plan, the allocation, the gaps and the ruling. That gives you an audit trail of how the ambition was shaped by constraints and policy before anyone said yes.

In [ ]:
final = result["final"]
print(f"Final decision after {final['round']} round(s): {final['decision']}\n")
print(final["plan"])

### Variations
- **Tool-backed Resources:** give the Resources role [tools](../../tool_use) that query real systems (budget APIs, calendars, inventory databases) instead of a static inventory string.
- **Human Authority:** replace the `authority` call with a human approval step. Vision and Resources still do the preparation, so the human reviews a plan that is already resourced.
- **Weighted corners:** use a stronger model for Authority and a cheaper, faster one for Vision drafts. Pass a different `model` to each role.